# Lab 4 — Hypothesis & Sampling

**Day 01 · Data Science Introduction · Cisco AI/ML Training**

---

## Learning objectives

1. Distinguish **population** vs **sample** statistics on the same CSV.
2. Frame a simple **hypothesis** (H₀ / H₁) about team growth.
3. Draw random samples and observe **sampling variation**.
4. Compute **proportions** (growth rate) overall and by region.
5. Build intuition for **confidence** via repeated samples (bootstrap preview).

> **Checkpoints:** pop mean ≈ **150.30** · sample(n=10) mean ≈ **132.60** · growth **0.75** · North **0.60**

<!-- cisco-day01-expanded-2026 -->

---

**Day 1 flow:** Lab 3 statistics → **Lab 4 (you are here)** → Lab 5 tool landscape → Lab 6 group Excel. Builds on descriptive stats from Lab 3.


## Why this matters

You rarely measure every customer or transaction. Sampling error is why A/B tests and loan default models need careful experimental design (Days 3–4).

## Hypothesis framing

| | Statement |
|--|----------|
| **H₀** | Most teams did **not** improve Q2 vs Q1 (growth rate ≤ 50%) |
| **H₁** | Majority of teams improved (growth rate > 50%) |

We observe **15/20 = 75%** with Q2 > Q1 — does that support H₁ in plain language?

---

## 1. Population statistics

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-01":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "hands-on" / "day-01" / "data" / "team_sales.csv").is_file():
            GH_ROOT = parent
            break

TEAM_SALES_CSV = GH_ROOT / "hands-on" / "day-01" / "data" / "team_sales.csv"
df = pd.read_csv(TEAM_SALES_CSV)

population_mean = df["q2_sales"].mean()
population_median = df["q2_sales"].median()
print(f"Population size N = {len(df)}")
print(f"Population mean Q2:   {population_mean:.2f}")
print(f"Population median Q2: {population_median:.2f}")


### 1b. Population growth flag

In [ ]:
df["grew"] = df["q2_sales"] > df["q1_sales"]
growth_count = int(df["grew"].sum())
growth_rate = df["grew"].mean()
print(f"Teams with Q2 > Q1: {growth_count} / {len(df)}")
print(f"Population growth rate: {growth_rate:.2f}")


---

## 2. Random sample (n = 10, random_state = 42)

In [ ]:
sample = df.sample(n=10, random_state=42)
sample_mean = sample["q2_sales"].mean()
diff = abs(population_mean - sample_mean)

print("Sample teams:")
display(sample[["team", "region", "q2_sales"]].sort_values("q2_sales"))
print(f"\nSample mean:      {sample_mean:.2f}")
print(f"Population mean:  {population_mean:.2f}")
print(f"Difference:       {diff:.2f}")
print("\nOne small sample can mislead — that is sampling error.")


### 2b. Sample growth rate vs population

In [ ]:
sample_growth = sample["grew"].mean()
print(f"Sample growth rate (n=10): {sample_growth:.2f}")
print(f"Population growth rate:    {growth_rate:.2f}")


---

## 3. Regional proportions — all four regions

In [ ]:
regional = (
    df.groupby("region")["grew"]
    .agg(growth_count="sum", teams="count")
    .assign(growth_rate=lambda x: x["growth_count"] / x["teams"])
    .round(3)
)
display(regional.sort_values("growth_rate", ascending=False))

north_growth = regional.loc["North", "growth_rate"]
print(f"North growth rate: {north_growth:.2f}")


---

## 4. Sampling distribution — 20 draws

In [ ]:
sample_means = [df.sample(n=10, random_state=i)["q2_sales"].mean() for i in range(20)]
means_series = pd.Series(sample_means, name="sample_mean")

print(f"20 sample means — min: {means_series.min():.1f}, max: {means_series.max():.1f}")
print(f"Mean of sample means: {means_series.mean():.2f}")
print(f"Spread (std):         {means_series.std(ddof=1):.2f}")

fig, ax = plt.subplots(figsize=(7, 3))
sns.histplot(means_series, bins=8, kde=True, ax=ax, color="teal")
ax.axvline(population_mean, color="red", linestyle="--", label="population mean")
ax.set_title("Sampling distribution of the mean (n=10, 20 draws)")
ax.legend()
plt.tight_layout()
plt.show()


### 4b. Ten draws side-by-side (extension)

In [ ]:
ten_draws = pd.DataFrame({
    f"draw_{i}": [df.sample(n=10, random_state=i)["q2_sales"].mean()] for i in range(10)
}).T
ten_draws.columns = ["sample_mean"]
display(ten_draws.round(2))
print(f"Range across 10 draws: {ten_draws['sample_mean'].min():.2f} – {ten_draws['sample_mean'].max():.2f}")


### 4c. Excel parallel — `RAND` / `INDEX` sampling

In Excel you can simulate sampling with helper columns:

1. `=RAND()` in a helper column to shuffle rows
2. Sort ascending on helper → take top 10 rows
3. `=AVERAGE()` on those 10 Q2 values

Re-sort (F9 recalc) and watch the average move — same lesson as the Python loop above.

### 4d. Stratified peek — sample 2 teams per region

In [ ]:
stratified = df.groupby("region", group_keys=False).apply(
    lambda g: g.sample(n=2, random_state=42)
)
print(f"Stratified sample size: {len(stratified)}")
display(stratified[["team", "region", "q2_sales"]].sort_values("region"))
print(f"Stratified mean Q2: {stratified['q2_sales'].mean():.2f}")
print(f"Simple random mean: {sample_mean:.2f}")


### 4e. Outlier impact on sample means (50 draws)

In [ ]:
draw_means = pd.Series(
    [df.sample(n=10, random_state=i)["q2_sales"].mean() for i in range(50)]
)
draws_with_13 = [
    "Team_13" in df.sample(n=10, random_state=i)["team"].values for i in range(50)
]
print(f"50 random samples (n=10): mean range {draw_means.min():.1f} – {draw_means.max():.1f}")
print(f"Team_13 appeared in {sum(draws_with_13)}/50 samples")
print(f"When Team_13 included, avg sample mean: {draw_means[draws_with_13].mean():.1f}")
print(f"When Team_13 excluded:                  {draw_means[[not x for x in draws_with_13]].mean():.1f}")


### 4f. Proportion practice — declining teams

In [ ]:
decline_rate = 1 - growth_rate
decliners = df.loc[~df["grew"], ["team", "region", "q1_sales", "q2_sales"]]
print(f"Decline rate: {decline_rate:.2f}")
display(decliners)


### 4g. Write-up worksheet (fill in your notebook or paper)

| Question | Your answer |
|----------|-------------|
| Population N | |
| Sample n (main demo) | |
| H₁ in one sentence | |
| Does 75% growth support H₁? | |
| Why not conclude causation? | |

---

## 5. Checkpoint

In [ ]:
assert len(df) == 20
assert abs(population_mean - 150.30) < 0.01
assert len(sample) == 10
assert abs(sample_mean - 132.60) < 0.01
assert growth_count == 15
assert abs(growth_rate - 0.75) < 0.001
assert abs(north_growth - 0.60) < 0.001
print("✓ All checkpoint assertions passed")


## Reflection questions

1. Why is the sample mean (132.60) below the population mean (150.30) for `random_state=42`?
2. If North's growth rate is 60%, can we say North is "better" than East (80%)? What is missing?
3. How would larger sample size (n=15) change the histogram spread?

**Previous:** [Lab 3 — Statistics basics](lab03_statistics_basics.ipynb)  
**Next:** [Lab 5 — Tool landscape](lab05_tool_landscape.ipynb)
